In [13]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import config


In [14]:
question = 'what software do data scientists use?'


In [15]:
embedding = OpenAIEmbeddings(model='text-embedding-ada-002',api_key=config.api_key)
vectorstore_from_directory = Chroma(
    persist_directory="./intro-to-ds-lectures",
    embedding_function=embedding)

In [16]:
retrieved_docs = vectorstore_from_directory.max_marginal_relevance_search(query=question,
                                                                k=5)

In [18]:
for i in retrieved_docs:
    print(f"Page Content {i.page_content} \n --------\nLecture Title:{i.metadata['Lecture Title']}\n" )

Page Content As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end 
 --------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to

In [19]:
retriever = vectorstore_from_directory.as_retriever(search_type='mmr',
                                                    search_kwargs ={
                                                        'k':3,
                                                        'lambda_mult':0.7
                                                    })

In [20]:
retriever 

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000020F449AFFA0>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.7})

In [21]:
question = 'what software do data scientists use?'

In [22]:
retrieved_docs = retriever.invoke(question)

In [23]:
for i in retrieved_docs:
    print(f"Page Content: {i.page_content}\n-------\nLecture Title:{i.metadata['Lecture Title']}\n")

Page Content: As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
-------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop distributes the computational tasks on multiple computers which is basically the way to handle big data nowadays. Power BI, SaS, Qlik, and especially Tableau are top-notch examples of software designed for business intelligence visualizations
-------
Lecture Title:Progra

at last now we'll be doing the generation step

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel
from langchain_core.output_parsers import StrOutputParser

In [24]:
len(vectorstore_from_directory.get()['documents'])

20

In [25]:
TEMPLATE = ''' 
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken form in the format:
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

In [28]:
prompt_template = ChatOpenAI(
    model_name='gpt-4',
    api_key=config.api_key,
    model_kwargs={
        'seed':365,
    },
    max_tokens = 250
)

c:\Users\user\miniconda3\envs\langchain_env\lib\site-packages\IPython\core\interactiveshell.py:3519: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


In [29]:
question = "what software do data scientists use?"

In [36]:
#construct the chain
chain = {'context':retriever,
         'question':RunnablePassthrough()} | prompt_template

In [37]:
chain.invoke(question)

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.